<a href="https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/ShaunGves/FlyRank-AI.git
%cd FlyRank-AI

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded successfully" if os.environ["HF_TOKEN"] else "Token missing")

Cloning into 'FlyRank-AI'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 129 (delta 41), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.83 MiB | 7.03 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/FlyRank-AI
Token loaded successfully


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, for one client, on one day (identified by client_hash_id + content_hash_id + report_date), from the fact_content_daily_performance table. Time window used for development: month=2026-03 — a mid-panel month, not the sealed _sample table (which is June 2026, reserved for final testing only).

In [2]:
import duckdb, os

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{}'
);
""".format(os.environ["HF_TOKEN"]))

query1 = """
SELECT COUNT(*) as total_rows,
       COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) as unique_combos
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.execute(query1).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_combos
0,9841378,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: content_type, main_intent, word_count, gsc_impressions, gsc_clicks, avg_position. Label/proxy: ai_sessions_90d and ai_traffic_pct. Context: client_hash_id, content_hash_id, report_date — identifiers, not signals. Excluded: health_score, priority_score, action_type — excluded because they're the product's own rule-based decisions, not raw observed signals; using them risks a circular result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
query2 = """
SELECT COUNT(*) as row_count, MIN(report_date) as earliest_date, MAX(report_date) as latest_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.execute(query2).df()

,row_count,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [4]:
query_cols = """
SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' LIMIT 1
"""
con.execute(query_cols).df().columns.tolist()


['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [5]:
query3 = """
SELECT COUNT(*) as total_rows,
       COUNT(sessions_ai) as rows_with_ai_value,
       SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) as rows_with_nonzero_ai_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.execute(query3).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ai_value,rows_with_nonzero_ai_sessions
0,9841378,6822637,5534.0


In [6]:
query3 = """
SELECT COUNT(*) as total_rows,
       COUNT(sessions_ai) as rows_with_ai_value,
       SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) as rows_with_nonzero_ai_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.execute(query3).df()

,total_rows,rows_with_ai_value,rows_with_nonzero_ai_sessions
0,9841378,6822637,5534.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Client history is unbalanced — different clients started tracking at different times (only 9 of 70 clients have 12+ months). Some early rows are GSC-only: before GA4 tracking started, ga4_data_available is False, so treating this as "zero traffic" would be wrong. This data can never prove causation — only correlation. Finally, feature windows must never overlap target windows, or results would be leaked and dishonestly optimistic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [Confirm ] Every section above is filled — markdown thinking AND the code that backs it
- [Confirm ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Confirm] No client names, URLs, or private queries anywhere
- [Confirm] My claims use careful words: observed, measured, directional, decision-support
- [Confirm ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.